In [0]:
# ================== Second take home assignment GR5069 Spring 2026 ========

# ================== Global Definitions ====================================

# Packages to import
from pyspark.sql.functions import avg, min, max, when, col, split, round, first, last, sum as spark_sum, rank, lit
from pyspark.sql.functions import to_date, current_date, datediff, floor, months_between
from pyspark.sql.functions import upper, substring, when, col, regexp_replace
from pyspark.sql.types import IntegerType
from pyspark.sql.window import Window

# Data Paths 
pitstops_path = "/Volumes/gr5069/raw/f1_data/pit_stops.csv"
results_path = "/Volumes/gr5069/raw/f1_data/results.csv"
drivers_path = "/Volumes/gr5069/raw/f1_data/drivers.csv"
races_path = "/Volumes/gr5069/raw/f1_data/races.csv"

##### 1. Average Pit Stop Time per Driver per Race
---------

**a. Load pitstop data**

Loaded pitstop table and defined as ```pit_stops``` dataframe. Converted columns from character to int types for analysis using following code. 

<br>

``` 
.withColumn().cast()) 
```

<br>

**b. Calculate average pitstop time**
<br>

Define dataframe ```pitstop_avg``` by: 

 Calculate a ```duration_seconds``` column by  converting the existing duration string column to a numeric value in seconds. 

Groups the data by driver and race using ```.groupBy("raceId", "driverId")```

Using ```.agg()``` calculate ```pitstop_avg_duration```,   ```fastest_pitstop```, ```slowest_pitstop```

using ```select()``` select necessary columns and then order the data by driverId and raceId ```.orderBy("driverId", "raceId")```


**c. Calculate slowest and fastest per race**

Define ```pitstop_race``` by:

using the ```pitstop_avg``` dataframe group by ```raceid``` and calculate the minimum ```pitstop_avg_duration``` for the fastest pitstop, the maximum ```pitstop_avg_duration``` as the slowest pitstop. 










In [0]:
# Load pitstop data
df_pitstops = spark.read.csv(pitstops_path, header=True) \
    .withColumn("raceId", col("raceId").cast("int")) \
    .withColumn("driverId", col("driverId").cast("int")) \
    .withColumn("lap", col("lap").cast("int")) \
    .withColumn("stop", col("stop").cast("int"))


In [0]:
# Calculate average pitstop time

pitstop_avg = df_pitstops.withColumn( 
    "duration_seconds",
    when(col("duration").contains(":"),
        split(col("duration"), ":")[0].cast("double") * 60 + split(col("duration"), ":")[1].cast("double")
    ).otherwise(col("duration").cast("double"))
).groupBy("raceId", "driverId") \
 .agg(
    round(avg("duration_seconds"), 4).alias("pitstop_avg_duration"),
    min("duration_seconds").alias("fastest_pitstop"),
    max("duration_seconds").alias("slowest_pitstop")
).select("driverId", "raceId", "pitstop_avg_duration", "fastest_pitstop", "slowest_pitstop") \
 .orderBy("driverId", "raceId")

display(pitstop_avg)

In [0]:
# Slowest and fastest per race
pitstop_race = pitstop_avg.groupBy("raceId") \
    .agg(
        min("pitstop_avg_duration").alias("fastest_pitstop"),
        max("pitstop_avg_duration").alias("slowest_pitstop")
    )

display(pitstop_race)

##### 2.  Rank order by finishing position the average time spent at the pit stop in each race.
---------


**a. Load results data**

Loaded results table and defined as ```df_results``` dataframe. Converted columns from character to int types for analysis using following code. 

<br>

``` 
.withColumn().cast()) 
```

**b. Calculate a pit stop ranked dataframe**

Define `pitstop_ranked` by joining `pitstop_avg` with `df_results` on `raceId` and `driverId` using a left join to include `positionOrder`, then selecting relevant columns and ordering by `raceId` and `positionOrder`.

**c. Join results table with pitstop table**

Define `pitstop_ranked` by selecting `raceId`, `driverId`, and `pitstop_avg_duration` from `pitstop_avg`. Create a `finishing_position` column using `.withColumn()` acting as an if-else statement. if `position` is null assign `"DNF"`, otherwise cast `positionOrder` to a string. Then join with `df_results` on `raceId` and `driverId` to include `finishing_position` and order by `raceId` and `finishing_position`.

This method ensures that even if a driver did not finish the race, they are included in the dataset but you can tell if they did not complete the race. 

In [0]:
# Load results data 
df_results = spark.read.csv(results_path, header=True) \
    .withColumn("raceId", col("raceId").cast("int")) \
    .withColumn("driverId", col("driverId").cast("int")) \
    .withColumn("positionOrder", col("positionOrder").cast("int")) \
    .withColumn("position", 
        when(col("position") == "\\N", None)
        .otherwise(col("position").cast("int"))
    )

display(df_results)

In [0]:
# Calculate a pit stop ranked dataframe 
pitstop_ranked = pitstop_avg.join(
    df_results.select("raceId", "driverId", "positionOrder"),
    on=["raceId", "driverId"],
    how="left"
).select("raceId", "positionOrder", "driverId", "pitstop_avg_duration") \
 .orderBy("raceId", "positionOrder")

display(pitstop_ranked)

In [0]:
# Join results table with pitstop table
pitstop_ranked = pitstop_avg.select("raceId", "driverId", "pitstop_avg_duration") \
    .join(
        df_results.withColumn("finishing_position",
            when(col("position").isNull(), "DNF")
            .otherwise(col("positionOrder").cast("string"))
        ).select("raceId", "driverId", "finishing_position"),
        on=["raceId", "driverId"],
        how="left"
    ).orderBy("raceId", "finishing_position")

display(pitstop_ranked)

In [0]:
# Check to see if there are any null finishing positions
pitstop_ranked.filter(col("finishing_position").isNull()).count()

##### 3. Insert the missing code (e.g: ALO for Alonso) for drivers based on the 'drivers' dataset. Explain your logic.
---------

**a. Load drivers data**

load drivers data and define it as ```df_drivers``` dataframe. 

**b. Add driver code for null values**

Define `df_drivers_add_code` by:

Using `.withColumn()` on `df_drivers`, update the `code` column — if `code` contains `"\\N"` (null), generate a code by taking the first 3 letters of the driver's `surname` and converting to uppercase using `upper(substring())`. Otherwise keep the existing `code` value.

**c. Alternative RegEx method**

Define `df_drivers_add_code` by:

Using `.withColumn()` calls on `df_drivers` use `regexp_replace()` to replace the `"\\N"` with an empty string. Then, if the resulting `code` is NA, replace that NA by taking the first 3 letters of the driver's `surname` and converting to uppercase using `upper(substring())`, otherwise keep the existing `code` value.

In [0]:
# Load drivers data 
df_drivers = spark.read.csv(drivers_path, header=True)
display(df_drivers)

In [0]:
# Add driver code for drivers who have null values in the "code" field 

df_drivers_add_code = df_drivers.withColumn(
    "code",
    when(col("code") == "\\N", upper(substring(col("surname"), 1, 3)))
    .otherwise(col("code"))
)
display(df_drivers_add_code)

In [0]:
# Check to see if there are any null values in the "code" field 
df_drivers_add_code.filter(col("code").isNull() | (col("code") == "\\N")).count()

In [0]:
# Alternative method: Use a regular expression to replace null values with the first three letters of the driver's surname. 

df_drivers_add_code = df_drivers \
    .withColumn("code", regexp_replace(col("code"), r"\\N", "")) \
    .withColumn("code", 
        when(col("code") == "", upper(substring(col("surname"), 1, 3)))
        .otherwise(col("code"))
    )

display(df_drivers_add_code)

In [0]:
# Check to see if there are any null values in the "code" field 
df_drivers_add_code.filter(col("code").isNull() | (col("code") == "\\N")).count()

##### 4. Who is the youngest and the oldest driver in each race? Create a new column called “Age”. Explain your definition of "age".
---------

**a. Load races data**

Loaded races table and defined as ```df_races``` dataframe. Converted columns to correct datatype for following analysis using: 

<br>

``` 
.withColumn().cast()) 
```

**b. Calculate youngest and oldest drivers per race**

Define `df_age` by selecting `raceId` and `driverId` from `df_results`, left joining with `df_races` to include race `date` and `df_drivers` to include `forename`, `surname`, and `dob`. Convert `dob` using `to_date()` and calculate driver `age` at race time using `months_between()` divided by 12 and rounded down with `floor()`.

Define `window_youngest` and `window_oldest` both partitioned by `raceId`, ordered by `dob` descending and ascending respectively.

Define `df_youngest` and `df_oldest` by ranking drivers within each race using each window, filtering to `rank == 1`, and adding a `category` label. Combine into `df_youngest_oldest` using `.union()` ordered by `raceId` and `category`.

In [0]:
# Load the races data
df_races = spark.read.csv(races_path, header=True) \
    .withColumn("raceId", col("raceId").cast("int")) \
    .withColumn("year", col("year").cast("int")) \
    .withColumn("date", to_date(col("date"), "yyyy-MM-dd"))

display(df_races.limit(5))

In [0]:
# Join results dataframe to races dataframe and drivers dataframe
df_age = df_results.select("raceId", "driverId") \
    .join(df_races.select("raceId", "date"), on="raceId", how="left") \
    .join(df_drivers.select("driverId", "forename", "surname", "dob"), on="driverId", how="left") \
    .withColumn("dob", to_date(col("dob"), "yyyy-MM-dd")) \
    .withColumn("age", floor(months_between(col("date"), col("dob")) / 12))

# Define window specifications to rank drivers by age within each race
window_youngest = Window.partitionBy("raceId").orderBy(col("dob").desc())
window_oldest = Window.partitionBy("raceId").orderBy(col("dob").asc())

# Rank drivers by age within each race and filter to the youngest and oldest driver per race
df_youngest = df_age.withColumn("rank", rank().over(window_youngest)).filter(col("rank") == 1).withColumn("category", lit("youngest"))
df_oldest = df_age.withColumn("rank", rank().over(window_oldest)).filter(col("rank") == 1).withColumn("category", lit("oldest"))

# Combine youngest and oldest dataframes and order by race and category
df_youngest_oldest = df_youngest.union(df_oldest).orderBy("raceId", "category")

display(df_youngest_oldest)

##### 5. At any given race, how many podiums does each driver have? create three new columns to show - on any given race - the number of wins, the number of 2nd places, and the number of 3rd places for each driver
---------
**a. Calculate cumulative podium finishes per driver**

Define `df_podiums` by selecting `raceId`, `driverId`, and `positionOrder` from `df_results`, left joining with `df_drivers_add_code` to include driver `code`, then creating three binary columns `is_win`, `is_2nd`, and `is_3rd` using `.withColumn()` assigning `1` if the driver finished in that position, `0` otherwise.

Define a window partitioned by `driverId` ordered by `raceId` with `rowsBetween(Window.unboundedPreceding, 0)` to compute a running cumulative total per driver up to each race.

Redefine `df_podiums` applying `spark_sum()` over the window to calculate `total_wins`, `total_2nd`, and `total_3rd`, then select relevant columns and order by `raceId` and `code`.

In [0]:
# Join results and drivers tables on driverId
df_podiums = df_results.select("raceId", "driverId", "positionOrder") \
    .join(df_drivers_add_code.select("driverId", "code"), on="driverId", how="left") \
# add columns for win, 2nd, and 3rd positions
    .withColumn("is_win", when(col("positionOrder") == 1, 1).otherwise(0)) \
    .withColumn("is_2nd", when(col("positionOrder") == 2, 1).otherwise(0)) \
    .withColumn("is_3rd", when(col("positionOrder") == 3, 1).otherwise(0))

# Define a window specification to calculate the total wins, 2nd, and 3rd positions for each driver
window = Window.partitionBy("driverId").orderBy("raceId").rowsBetween(Window.unboundedPreceding, 0)

# Calculate the total wins, 2nd, and 3rd positions for each driver
df_podiums = df_podiums \
    .withColumn("total_wins", spark_sum("is_win").over(window)) \
    .withColumn("total_2nd", spark_sum("is_2nd").over(window)) \
    .withColumn("total_3rd", spark_sum("is_3rd").over(window)) \
    .select("raceId", "code", "driverId", "total_wins", "total_2nd", "total_3rd") \
    .orderBy("raceId", "code")

display(df_podiums)

##### 6. Continue exploring the data by answering your own question.
-----------

Calculate top ten drivers by total wins 

**a. Calculate top ten drivers by total wins**

Define ```df_podium_top_ten``` by grouping ```df_podiums``` by ```driverId``` and ```code```, aggregating to get the maximum total_wins per driver using ```max()```, and ordering by total_wins descending. Display the top 10 results using ```.limit(10)```.

In [0]:
# Calculate the total wins for each driver
df_podium_top_ten = df_podiums.groupBy("driverId", "code") \
    .agg(max("total_wins").alias("total_wins")) \
    .orderBy(col("total_wins").desc()) 

display(df_podium_top_ten.limit(10))